# Random Forest + Linear Regression (Mini-batch Gradient Descent) with Voting

We train 2 models on the same data and combine them with **voting** (the average of their predictions).

- Model 1: **Random Forest**
- Model 2: **Linear Regression** trained with **Mini-batch Gradient Descent**, without and with momentum

Mini-batch GD updates the weights after every small batch (32 samples). With momentum we use lr × (1 − 0.9).

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras.layers import Dense, Input
from keras.models import Sequential
from keras.callbacks import Callback
from keras.optimizers import SGD, RMSprop, Adam
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

keras.utils.set_random_seed(42)

## Data
**Consumer Airfare Report (Table 6)**, U.S. Department of Transportation: every route between two US cities, every quarter, since the 1990s.

- target: `fare`, the average one-way ticket price of the route (regression)
- features: distance, log of passengers per day, market share of the biggest and of the cheapest airline, year, quarter, and which airline is the biggest and the cheapest (one-hot)
- we did not use `fare_lg` and `fare_low` because they are prices (data leakage)
- test set = the last 2 years, train set = all the years before
- we use a random sample of 50,000 rows so the training is faster

In [ ]:
url = "https://data.transportation.gov/resource/yj5y-b2ir.csv?$select=year,quarter,nsmiles,passengers,fare,large_ms,lf_ms,carrier_lg,carrier_low&$limit=2000000"
try:
    df = pd.read_csv(url)
except:
    df = pd.read_csv("https://data.transportation.gov/api/views/yj5y-b2ir/rows.csv?accessType=DOWNLOAD", low_memory=False)

df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
print(df.shape)
df.head()

In [ ]:
df = df[["year", "quarter", "nsmiles", "passengers", "fare", "large_ms", "lf_ms", "carrier_lg", "carrier_low"]].copy()
num = ["year", "quarter", "nsmiles", "passengers", "fare", "large_ms", "lf_ms"]
df[num] = df[num].apply(pd.to_numeric, errors="coerce")
df = df.dropna()
df = df[df["fare"].between(20, 2000) & (df["nsmiles"] > 0) & (df["passengers"] > 0)]
df = df[df["large_ms"].between(0, 1) & df["lf_ms"].between(0, 1)]
df = df.sample(n=min(50000, len(df)), random_state=42)

for c in ["carrier_lg", "carrier_low"]:
    top = df[c].value_counts().index[:8]
    df[c] = df[c].where(df[c].isin(top), "Other")

df["log_passengers"] = np.log(df["passengers"])
df = pd.get_dummies(df, columns=["quarter", "carrier_lg", "carrier_low"], drop_first=True, dtype=int)

feature_cols = ["nsmiles", "log_passengers", "large_ms", "lf_ms", "year"]
feature_cols += [c for c in df.columns if c.startswith(("quarter_", "carrier_lg_", "carrier_low_"))]
print(df.shape, "features:", len(feature_cols))

We scale the features and also the fare for the linear model. After training we convert its predictions back to dollars.

In [ ]:
last_year = df["year"].max()
train = df[df["year"] <= last_year - 2]
test = df[df["year"] > last_year - 2]

X_train = train[feature_cols].values.astype(float)
X_test = test[feature_cols].values.astype(float)
y_train = train["fare"].values
y_test = test["fare"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

y_mean, y_std = y_train.mean(), y_train.std()
y_train_s = (y_train - y_mean) / y_std

print("train:", X_train_s.shape, "test:", X_test_s.shape)

## Best possible linear model (Normal Equation)
Linear regression has an exact solution. The optimizers are trying to reach it.

In [ ]:
Xb = np.hstack([X_train_s, np.ones((len(X_train_s), 1))])
w_best = np.linalg.lstsq(Xb, y_train_s, rcond=None)[0]
best_loss = np.mean((Xb @ w_best - y_train_s) ** 2)

Xb_test = np.hstack([X_test_s, np.ones((len(X_test_s), 1))])
ne_preds = Xb_test @ w_best * y_std + y_mean

print("best possible train loss:", round(best_loss, 4))
print("Normal Equation test R2:", round(r2_score(y_test, ne_preds), 4))

## Model 1: Random Forest
Many decision trees, each one trained on a random sample of the data. The forest predicts the average of the trees. It is not trained with gradient descent.

In [ ]:
start = time.time()
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_time = time.time() - start
rf_preds = rf.predict(X_test)

print("Random Forest test R2:", round(r2_score(y_test, rf_preds), 4), " time:", round(rf_time, 1), "s")

## Model 2: Linear Regression with Mini-batch Gradient Descent
The linear model is `Dense(1)`, trained on the MSE loss.

In [ ]:
EPOCHS = 20
LR = 0.01
BATCH_SIZE = 32
LOG_EVERY = 100

In [ ]:
class Logger(Callback):
    def on_train_begin(self, logs=None):
        self.step = 0
        self.start = time.time()
        self.losses, self.epochs, self.times, self.path = [], [], [], [[0.0, 0.0]]

    def on_train_batch_end(self, batch, logs=None):
        self.step += 1
        if self.step % LOG_EVERY == 0:
            w, b = self.model.layers[0].get_weights()
            pred = X_train_s @ w[:, 0] + b[0]
            self.losses.append(np.mean((pred - y_train_s) ** 2))
            self.epochs.append(self.step * BATCH_SIZE / len(X_train_s))
            self.times.append(time.time() - self.start)
            self.path.append([w[0, 0], w[1, 0]])


def train_model(lr, momentum=0.0):
    keras.utils.set_random_seed(42)
    model = Sequential([Input(shape=(X_train_s.shape[1],)), Dense(1, kernel_initializer="zeros")])
    model.compile(optimizer=SGD(learning_rate=lr, momentum=momentum), loss="mse")
    log = Logger()
    start = time.time()
    model.fit(X_train_s, y_train_s, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=[log])
    log.time = time.time() - start
    log.preds = model.predict(X_test_s, verbose=0).flatten() * y_std + y_mean
    log.final_loss = log.losses[-1]
    print(f"done in {log.time:.1f} s, final train loss = {log.final_loss:.4f}")
    return log

In [ ]:
run_1 = train_model(LR)
run_2 = train_model(LR * (1 - 0.9), momentum=0.9)

## Voting and results

In [ ]:
names = ["Mini-batch GD", "Mini-batch GD + Momentum"]
runs = [run_1, run_2]

rows = []
for name, run in zip(names, runs):
    vote = (run.preds + rf_preds) / 2
    rows.append({"model": name, "gap to best (%)": (run.final_loss - best_loss) / best_loss * 100,
                 "test R2": r2_score(y_test, run.preds),
                 "test RMSE ($)": np.sqrt(mean_squared_error(y_test, run.preds)), "time (s)": run.time})
    rows.append({"model": "Voting: RF + " + name, "gap to best (%)": np.nan,
                 "test R2": r2_score(y_test, vote),
                 "test RMSE ($)": np.sqrt(mean_squared_error(y_test, vote)), "time (s)": run.time + rf_time})
    run.vote = vote

rows.append({"model": "Random Forest", "gap to best (%)": np.nan, "test R2": r2_score(y_test, rf_preds),
             "test RMSE ($)": np.sqrt(mean_squared_error(y_test, rf_preds)), "time (s)": rf_time})
rows.append({"model": "Normal Equation (linear)", "gap to best (%)": 0.0, "test R2": r2_score(y_test, ne_preds),
             "test RMSE ($)": np.sqrt(mean_squared_error(y_test, ne_preds)), "time (s)": np.nan})

table = pd.DataFrame(rows).round(4)
print(table.to_string(index=False))

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
for name, run in zip(names, runs):
    plt.plot(run.epochs, run.losses, label=name)
plt.axhline(best_loss, color="red", linestyle="--", label="Normal Equation")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("train MSE (scaled fare)")
plt.title("Training loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
for name, run in zip(names, runs):
    gap = (np.array(run.losses) - best_loss) / best_loss * 100
    plt.plot(run.times, np.maximum(gap, 1e-6), label=name)
plt.yscale("log")
plt.xlabel("training time (s)")
plt.ylabel("gap to the best loss (%)")
plt.title("Gap to the best solution")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("rf_minibatch_loss.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
labels = ["Linear\n" + names[0], "Linear\n" + names[1], "Random\nForest", "Voting\n" + names[0], "Voting\n" + names[1]]
values = [r2_score(y_test, runs[0].preds), r2_score(y_test, runs[1].preds), r2_score(y_test, rf_preds),
          r2_score(y_test, runs[0].vote), r2_score(y_test, runs[1].vote)]

plt.figure(figsize=(11, 5))
bars = plt.bar(labels, values, color=["tab:blue", "tab:blue", "tab:green", "tab:orange", "tab:orange"])
for b, v in zip(bars, values):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.005, f"{v:.3f}", ha="center")
plt.ylim(min(values) - 0.1, max(values) + 0.05)
plt.ylabel("test R2")
plt.title("Linear vs Random Forest vs Voting")
plt.savefig("rf_minibatch_r2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
idx = np.random.choice(len(y_test), min(3000, len(y_test)), replace=False)
best_run = max(runs, key=lambda r: r2_score(y_test, r.vote))

plt.figure(figsize=(15, 4.5))
for k, (title, p) in enumerate([("Linear", best_run.preds), ("Random Forest", rf_preds), ("Voting", best_run.vote)]):
    plt.subplot(1, 3, k + 1)
    plt.scatter(y_test[idx], p[idx], s=5, alpha=0.4)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    plt.xlabel("actual fare ($)")
    plt.ylabel("predicted fare ($)")
    plt.title(f"{title} (R2 = {r2_score(y_test, p):.3f})")
plt.tight_layout()
plt.savefig("rf_minibatch_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## Path on the loss surface
Only 2 weights of the linear model are drawn (distance and log passengers). The other weights are fixed at their best values.

In [ ]:
Xs, ys = X_train_s[::10], y_train_s[::10]
w0 = w_best[:-1]

def loss_at(a, b):
    w = w0.copy()
    w[0] = a
    w[1] = b
    return np.mean((Xs @ w + w_best[-1] - ys) ** 2)

paths = [np.array(run.path) for run in runs]
points = np.vstack(paths + [w0[None, :2]])
low, high = points.min(axis=0), points.max(axis=0)
pad = 0.15 * (high - low) + 0.05
A, B = np.meshgrid(np.linspace(low[0] - pad[0], high[0] + pad[0], 40),
                   np.linspace(low[1] - pad[1], high[1] + pad[1], 40))
Z = np.log10(np.array([[loss_at(a, b) for a, b in zip(ra, rb)] for ra, rb in zip(A, B)]))

colors = ["black", "magenta"]
fig = plt.figure(figsize=(15, 6))

ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(A, B, Z, cmap="coolwarm", alpha=0.6)
for p, name, c in zip(paths, names, colors):
    ax.plot(p[:, 0], p[:, 1], [np.log10(loss_at(a, b)) for a, b in p], color=c, marker=".", label=name)
ax.scatter(w0[0], w0[1], Z.min(), color="red", marker="*", s=200, label="best solution")
ax.set_xlabel("weight of distance")
ax.set_ylabel("weight of log passengers")
ax.set_zlabel("log10(loss)")
ax.set_title("Loss surface")
ax.legend()

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contourf(A, B, Z, levels=25, cmap="coolwarm")
plt.colorbar(cs, ax=ax2, label="log10(loss)")
for p, name, c in zip(paths, names, colors):
    ax2.plot(p[:, 0], p[:, 1], color=c, marker=".", markersize=4, label=name)
ax2.plot(0, 0, "go", markersize=10, label="start")
ax2.plot(w0[0], w0[1], "r*", markersize=18, label="best solution")
ax2.set_xlabel("weight of distance")
ax2.set_ylabel("weight of log passengers")
ax2.set_title("View from above")
ax2.legend()

plt.tight_layout()
plt.savefig("rf_minibatch_path.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
out = {"best_loss": float(best_loss),
       "rf": {"r2": float(r2_score(y_test, rf_preds)), "time": float(rf_time)},
       "ne_r2": float(r2_score(y_test, ne_preds)),
       "runs": [{"name": name, "losses": [float(v) for v in run.losses], "epochs": run.epochs, "times": run.times,
                 "gap": float((run.final_loss - best_loss) / best_loss * 100), "time": float(run.time),
                 "r2": float(r2_score(y_test, run.preds)), "vote_r2": float(r2_score(y_test, run.vote)),
                 "vote_rmse": float(np.sqrt(mean_squared_error(y_test, run.vote)))}
                for name, run in zip(names, runs)]}

with open("results_rf_minibatch.json", "w") as f:
    json.dump(out, f)